In [1]:
import pandas as pd
import numpy as np
import requests
import os
import datetime as dt
import re
import ast

pd.set_option('display.max_columns', 999)

import warnings
warnings.filterwarnings('ignore')

In [2]:
#Helper Functions

def convert_to_num(df):
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
        except:
            print(f"Column {col} cannot be converted to numeric")
            continue

def labelDom(row):
    if row['DOM'] < 14:
        return True
    else:
        return False

def binaryDom(df):
    df['binaryDOM'] = df.apply(lambda row: labelDom(row), axis = 1)

def createDOM(df: pd.DataFrame, timestamp = False, convert = False):
    dom = df['soldDate'] - df['listDate']
    df['DOM'] = dom.astype("string")
    df.dropna(subset=['DOM'], inplace = True)
    df['DOM'] = df['DOM'].str.slice_replace(start=2)
    df['DOM'] = df['DOM'].astype("int")

def dfRemoveNA(df):
    df = df.replace('', np.nan)
    if sum(df.isna().sum()) != 0:
        df = df.dropna()
    return df

def dfFillNA(df, column):
    df = df.replace('', np.nan)
    df = df.fillna(value = {column: "No "+column})
    return df

def dfExtractFeatures(df: pd.DataFrame, key: str, columns: list):
    if type(df[key].values[0]) == str:
        try:
            serie = df[key].apply(lambda x: ast.literal_eval(str(x)))
            d = pd.DataFrame.from_dict(serie.tolist())
        except e:
            print("String does not match proper conversion methods")
    elif type(df[key].values[0]) == dict:
        serie = pd.Series.to_dict(df[key])
        d = pd.DataFrame.from_dict(serie, orient='index')
    for i in range(len(columns)):
        df[columns[i]] = d[columns[i]]

    return df

def dfPriceDifference(df: pd.DataFrame):
    #should only be for clustering purposes only
    df['PriceDifference'] = df['soldPrice'] - df['listPrice']

def dfDatesRemoveTimeStamp(df: pd.DataFrame):
    df['listDate'] = df['listDate'].str.slice_replace(start = 10, repl="")
    df['soldDate'] = df['soldDate'].str.slice_replace(start = 10, repl="")

def dfConvertToDatetime(df: pd.DataFrame, timestamp = False):
    if type(df['listDate'].values[0]) == str:
        dfDatesRemoveTimeStamp(df)
    df['listDate'] = pd.to_datetime(df['listDate'])
    df['soldDate'] = pd.to_datetime(df['soldDate'])

def removeFeatures(df: pd.DataFrame, columns: list):
    df = df.drop(columns, axis = 1)
    return df

#Data Augmentation

def countLevels(df: pd.DataFrame, column):
    series = df[column].value_counts()
    return series

def augmentCategorical(df, column, percent = 0.04):
    series = countLevels(df, column)
    total = np.sum(series.values)
    percentage = [x/total for x in series.values]
    
    for i in range(len(percentage)):
        if percentage[i] > percent:
            pass
        else:
            df = df.replace(to_replace = {column: series.index[i]}, value = 'Other '+column)

    return df

def augmentType(df, type, columns):
    for col in columns:
        df[col] = df[col].astype(df[col].astype(type))

    return df

def augmentNumericals(df):
    for i in range(len(df.columns)):
        if (type(df[df.columns[i]].values[0]) == str):
            numerical = re.search('^[-+]?[0-9]*\.?[0-9]+$', df[df.columns[i]].values[0])
            if numerical is not None:
                df[df.columns[i]] = df[df.columns[i]].astype(float)
    return df

def removeLowLevels(df, column):
    lst = []
    for i in range(len(df[column].value_counts().index)):
        if df[column].value_counts().values[i] < 11:
            lst.append(df[column].value_counts().index[i])

    for i in range(len(lst)):
        df = df[df[column] != lst[i]]
    return df

def removeOutliers(df):
    percentile90 = df['soldPrice'].quantile(0.90)
    percentile10 = df['soldPrice'].quantile(0.10)

    iqr = percentile90 - percentile10

    upper_limit = percentile90 + 3 * iqr
    lower_limit = percentile10 - 3 * iqr

    # df[df['soldPrice'] > upper_limit]
    # df[df['soldPrice'] < lower_limit]

    df = df[df['soldPrice'] < upper_limit]
    df = df[df['soldPrice'] > lower_limit]

    return df


Lease

In [3]:
raw_lease_df = pd.read_csv(os.getcwd()+"\\data\\raw_lease_data.csv")

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\1015541478.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_lease_df = pd.read_csv(os.getcwd()+"\\data\\raw_lease_data.csv")


In [ ]:
raw_lease_df.shape

In [4]:
raw_lease_df.head()

,listDate,originalPrice,soldDate,type,mlsNumber,soldPrice,details,class,map,address,listPrice,lastStatus,status,raw,bathrooms
0,2019-05-14T21:00:00.000Z,1950.0,2019-05-15T21:00:00.000Z,Lease,C4451062,1950.0,"{'numKitchens': '1', 'numParkingSpaces': '1.0'...",CondoProperty,"{'latitude': '43.7655092', 'point': 'POINT (-7...","{'area': 'Toronto', 'zip': 'M2N7C6', 'country'...",1950.0,Lsd,U,NaN,NaN
1,2019-04-06T00:00:00.000Z,2100.0,2019-05-15T00:00:00.000Z,Lease,N4406802,2030.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '44.2937029', 'point': 'POINT (-7...","{'area': 'Simcoe', 'zip': 'L9S 0N5', 'country'...",2030.0,Lsd,U,NaN,NaN
2,2019-04-25T00:00:00.000Z,2900.0,2019-05-15T00:00:00.000Z,Lease,W4427957,2900.0,"{'numKitchens': None, 'numParkingSpaces': '2.0...",ResidentialProperty,"{'latitude': '43.4939838', 'point': 'POINT (-7...","{'area': 'Halton', 'zip': 'L6H7H5', 'country':...",2900.0,Lsd,U,NaN,NaN
3,2019-05-13T00:00:00.000Z,2300.0,2019-05-15T00:00:00.000Z,Lease,E4448081,2300.0,"{'numKitchens': None, 'numParkingSpaces': '3.0...",ResidentialProperty,"{'latitude': '43.7171172', 'point': 'POINT (-7...","{'area': 'Toronto', 'zip': 'M1L 04J', 'country...",2300.0,Lsd,U,NaN,NaN
4,2019-05-11T00:00:00.000Z,2650.0,2019-05-15T00:00:00.000Z,Lease,N4447592,2650.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '43.8575305', 'point': 'POINT (-7...","{'area': 'York', 'zip': 'L4H4V2', 'country': N...",2650.0,Lsd,U,NaN,NaN


In [5]:
raw_lease_df.columns

Index(['listDate', 'originalPrice', 'soldDate', 'type', 'mlsNumber',
       'soldPrice', 'details', 'class', 'map', 'address', 'listPrice',
       'lastStatus', 'status', 'raw', 'bathrooms'],
      dtype='object')

The datatypes are mainly objects, but there are dictionaries and floats/integers that are located inside each objects that may need to be converted. Certain predictors like number of bathrooms can be left as strings, 

In [8]:
raw_lease_df.dtypes

listDate          object
originalPrice    float64
soldDate          object
type              object
mlsNumber         object
soldPrice        float64
details           object
class             object
map               object
address           object
listPrice        float64
lastStatus        object
status            object
raw              float64
bathrooms         object
dtype: object

In [9]:
address = ['area', 'city', 'district', 'neighborhood']
details = ['numBathrooms','numBedrooms','sqft','style',]
map_ = ['latitude', 'longitude']

lease_df = dfExtractFeatures(raw_lease_df, key = 'address', columns = address)
lease_df = dfExtractFeatures(lease_df, key = 'details', columns = details)
lease_df = dfExtractFeatures(lease_df, key = 'map', columns = map_)

In [10]:
lease_df.head()

,listDate,originalPrice,soldDate,type,mlsNumber,soldPrice,details,class,map,address,listPrice,lastStatus,status,raw,bathrooms,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude
0,2019-05-14T21:00:00.000Z,1950.0,2019-05-15T21:00:00.000Z,Lease,C4451062,1950.0,"{'numKitchens': '1', 'numParkingSpaces': '1.0'...",CondoProperty,"{'latitude': '43.7655092', 'point': 'POINT (-7...","{'area': 'Toronto', 'zip': 'M2N7C6', 'country'...",1950.0,Lsd,U,NaN,NaN,Toronto,Toronto,Toronto C07,Lansing-Westgate,1,2,700-799,Apartment,43.7655092,-79.4140924
1,2019-04-06T00:00:00.000Z,2100.0,2019-05-15T00:00:00.000Z,Lease,N4406802,2030.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '44.2937029', 'point': 'POINT (-7...","{'area': 'Simcoe', 'zip': 'L9S 0N5', 'country'...",2030.0,Lsd,U,NaN,NaN,Simcoe,Innisfil,Innisfil,Alcona,3,4,2500-3000,2-Storey,44.2937029,-79.5436498
2,2019-04-25T00:00:00.000Z,2900.0,2019-05-15T00:00:00.000Z,Lease,W4427957,2900.0,"{'numKitchens': None, 'numParkingSpaces': '2.0...",ResidentialProperty,"{'latitude': '43.4939838', 'point': 'POINT (-7...","{'area': 'Halton', 'zip': 'L6H7H5', 'country':...",2900.0,Lsd,U,NaN,NaN,Halton,Oakville,Oakville,Rural Oakville,4,4,2500-3000,2-Storey,43.4939838,-79.7160835
3,2019-05-13T00:00:00.000Z,2300.0,2019-05-15T00:00:00.000Z,Lease,E4448081,2300.0,"{'numKitchens': None, 'numParkingSpaces': '3.0...",ResidentialProperty,"{'latitude': '43.7171172', 'point': 'POINT (-7...","{'area': 'Toronto', 'zip': 'M1L 04J', 'country...",2300.0,Lsd,U,NaN,NaN,Toronto,Toronto,Toronto E04,Clairlea-Birchmount,3,2,,3-Storey,43.7171172,-79.28078119999999
4,2019-05-11T00:00:00.000Z,2650.0,2019-05-15T00:00:00.000Z,Lease,N4447592,2650.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '43.8575305', 'point': 'POINT (-7...","{'area': 'York', 'zip': 'L4H4V2', 'country': N...",2650.0,Lsd,U,NaN,NaN,York,Vaughan,Vaughan,Kleinburg,3,4,3000-3500,2-Storey,43.8575305,-79.6185951


Feature Engineering

In [11]:
lease_df = lease_df[lease_df['soldPrice'] != 0]

In [12]:
lease_df.replace({"": np.nan}, inplace = True)
lease_df.replace({float("nan"): np.nan}, inplace = True)

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3185690225.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lease_df.replace({"": np.nan}, inplace = True)
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3185690225.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lease_df.replace({float("nan"): np.nan}, inplace = True)


In [13]:
dfConvertToDatetime(lease_df)

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['listDate'] = df['listDate'].str.slice_replace(start = 10, repl="")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['soldDate'] = df['soldDate'].str.slice_replace(start = 10, repl="")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

In [14]:
createDOM(lease_df)

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DOM'] = dom.astype("string")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(subset=['DOM'], inplace = True)
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https

In [15]:
lease_df['avg_sqft'] = [(int(value.split("-")[0])+int(value.split("-")[1]))/2 if not isinstance(value, float) and '-' in value else np.nan for value in lease_df['sqft']]

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\1208874281.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lease_df['avg_sqft'] = [(int(value.split("-")[0])+int(value.split("-")[1]))/2 if not isinstance(value, float) and '-' in value else np.nan for value in lease_df['sqft']]


In [16]:
lease_df['avg_sqft']

0          749.5
1         2750.0
2         2750.0
3            NaN
4         3250.0
           ...  
323249     949.5
323250    2250.0
323251     649.5
323252       NaN
323253    1099.5
Name: avg_sqft, Length: 323252, dtype: float64

In [17]:
lease_df['ppsqft'] = lease_df['listPrice']/lease_df['avg_sqft']

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\140138033.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lease_df['ppsqft'] = lease_df['listPrice']/lease_df['avg_sqft']


In [18]:
lease_df['ppsqft']

0         2.601734
1         0.738182
2         1.054545
3              NaN
4         0.815385
            ...   
323249    4.212744
323250    1.244444
323251    3.695150
323252         NaN
323253    2.910414
Name: ppsqft, Length: 323252, dtype: float64

In [19]:
lease_df['bathtobed_ratio'] = lease_df['numBathrooms'].astype("Int64")/lease_df['numBedrooms'].astype("Int64")

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\580071447.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lease_df['bathtobed_ratio'] = lease_df['numBathrooms'].astype("Int64")/lease_df['numBedrooms'].astype("Int64")


In [20]:
lease_df['latitude'] = lease_df['latitude'].astype("float")
lease_df['longitude'] = lease_df['longitude'].astype("float")

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\1096142183.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lease_df['latitude'] = lease_df['latitude'].astype("float")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\1096142183.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lease_df['longitude'] = lease_df['longitude'].astype("float")


In [21]:
avg_price_by_area = lease_df.groupby('area').agg('mean')['listPrice']
avg_price_by_city = lease_df.groupby('city').agg('mean')['listPrice']

In [22]:
lease_df_agg = lease_df.join(avg_price_by_area, on = 'area', rsuffix='_by_area')
lease_df_agg = lease_df_agg.join(avg_price_by_city, on = 'city', rsuffix='_by_city')

In [23]:
lease_df_agg.columns

Index(['listDate', 'originalPrice', 'soldDate', 'type', 'mlsNumber',
       'soldPrice', 'details', 'class', 'map', 'address', 'listPrice',
       'lastStatus', 'status', 'raw', 'bathrooms', 'area', 'city', 'district',
       'neighborhood', 'numBathrooms', 'numBedrooms', 'sqft', 'style',
       'latitude', 'longitude', 'DOM', 'avg_sqft', 'ppsqft', 'bathtobed_ratio',
       'listPrice_by_area', 'listPrice_by_city'],
      dtype='object')

In [24]:
columns_to_drop = ['listDate', 'address', 'soldDate', 'type', 'mlsNumber','details', 'map' ,'lastStatus', 'status', 'raw', 'bathrooms']

In [25]:
lease_df_agg = lease_df_agg.drop(columns = columns_to_drop, axis = 1)

In [26]:
#Remove any outliers
lease_df_agg = removeOutliers(lease_df_agg)

In [27]:
lease_df_agg.head()

,originalPrice,soldPrice,class,listPrice,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio,listPrice_by_area,listPrice_by_city
0,1950.0,1950.0,CondoProperty,1950.0,Toronto,Toronto,Toronto C07,Lansing-Westgate,1,2,700-799,Apartment,43.765509,-79.414092,1,749.5,2.601734,0.5,2556.449239,2556.449239
1,2100.0,2030.0,ResidentialProperty,2030.0,Simcoe,Innisfil,Innisfil,Alcona,3,4,2500-3000,2-Storey,44.293703,-79.543650,39,2750.0,0.738182,0.75,2505.232627,2827.100517
2,2900.0,2900.0,ResidentialProperty,2900.0,Halton,Oakville,Oakville,Rural Oakville,4,4,2500-3000,2-Storey,43.493984,-79.716083,20,2750.0,1.054545,1.0,2990.819872,3258.411820
3,2300.0,2300.0,ResidentialProperty,2300.0,Toronto,Toronto,Toronto E04,Clairlea-Birchmount,3,2,NaN,3-Storey,43.717117,-79.280781,2,NaN,NaN,1.5,2556.449239,2556.449239
4,2650.0,2650.0,ResidentialProperty,2650.0,York,Vaughan,Vaughan,Kleinburg,3,4,3000-3500,2-Storey,43.857531,-79.618595,4,3250.0,0.815385,0.75,2608.101053,2604.451783


Encoding to categorical variables

In [28]:
lease_df_agg_cats = pd.get_dummies(lease_df_agg)

In [29]:
lease_df_agg_cats.shape

(322448, 1337)

In [30]:
lease_df_agg_cats.to_csv(os.getcwd()+"\\data\\lease_data.csv", index = False)

Sale

In [31]:
raw_sale_df = pd.read_csv(os.getcwd()+"\\data\\raw_sale_data.csv")

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\746810782.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_sale_df = pd.read_csv(os.getcwd()+"\\data\\raw_sale_data.csv")


In [36]:
raw_sale_df.columns

In [ ]:
raw_sale_df.shape

In [37]:
raw_sale_df.head()

,listDate,originalPrice,soldDate,type,mlsNumber,soldPrice,details,class,map,address,listPrice,lastStatus,status,raw,bathrooms
0,2019-04-01T00:00:00.000Z,1499000.0,2019-05-15T00:00:00.000Z,Sale,N4400080,1480000.0,"{'numKitchens': None, 'numParkingSpaces': '5.0...",ResidentialProperty,"{'latitude': '43.8797315', 'point': 'POINT (-7...","{'area': 'York', 'zip': 'L6A0L4', 'country': N...",1499000.0,Sld,U,NaN,NaN
1,2019-04-09T00:00:00.000Z,649900.0,2019-05-15T00:00:00.000Z,Sale,N4409516,615000.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '44.31391250000001', 'point': 'PO...","{'area': 'Simcoe', 'zip': 'L0M1B6', 'country':...",634900.0,Sld,U,NaN,NaN
2,2019-05-01T00:00:00.000Z,929900.0,2019-05-15T00:00:00.000Z,Sale,W4434683,885000.0,"{'numKitchens': None, 'numParkingSpaces': '6.0...",ResidentialProperty,"{'latitude': '43.734903', 'point': 'POINT (-79...","{'area': 'Peel', 'zip': 'L6R2H6', 'country': N...",899000.0,Sld,U,NaN,NaN
3,2019-05-08T00:00:00.000Z,848800.0,2019-05-15T00:00:00.000Z,Sale,W4442890,841000.0,"{'numKitchens': None, 'numParkingSpaces': '6.0...",ResidentialProperty,"{'latitude': '43.6223313', 'point': 'POINT (-7...","{'area': 'Peel', 'zip': 'L4Z3T8', 'country': N...",848800.0,Sld,U,NaN,NaN
4,2019-04-24T00:00:00.000Z,692000.0,2019-05-15T00:00:00.000Z,Sale,N4426576,663000.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '43.8893334', 'point': 'POINT (-7...","{'area': 'York', 'zip': 'L4C2X7', 'country': N...",692000.0,Sld,U,NaN,NaN


In [38]:
raw_sale_df.columns

Index(['listDate', 'originalPrice', 'soldDate', 'type', 'mlsNumber',
       'soldPrice', 'details', 'class', 'map', 'address', 'listPrice',
       'lastStatus', 'status', 'raw', 'bathrooms'],
      dtype='object')

In [39]:
raw_sale_df.dtypes

listDate          object
originalPrice    float64
soldDate          object
type              object
mlsNumber         object
soldPrice        float64
details           object
class             object
map               object
address           object
listPrice        float64
lastStatus        object
status            object
raw              float64
bathrooms         object
dtype: object

In [40]:
address = ['area', 'city', 'district', 'neighborhood']
details = ['numBathrooms','numBedrooms','sqft','style',]
map_ = ['latitude', 'longitude']

sale_df = dfExtractFeatures(raw_sale_df, key = 'address', columns = address)
sale_df = dfExtractFeatures(sale_df, key = 'details', columns = details)
sale_df = dfExtractFeatures(sale_df, key = 'map', columns = map_)

In [41]:
sale_df.head()

,listDate,originalPrice,soldDate,type,mlsNumber,soldPrice,details,class,map,address,listPrice,lastStatus,status,raw,bathrooms,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude
0,2019-04-01T00:00:00.000Z,1499000.0,2019-05-15T00:00:00.000Z,Sale,N4400080,1480000.0,"{'numKitchens': None, 'numParkingSpaces': '5.0...",ResidentialProperty,"{'latitude': '43.8797315', 'point': 'POINT (-7...","{'area': 'York', 'zip': 'L6A0L4', 'country': N...",1499000.0,Sld,U,NaN,NaN,York,Vaughan,Vaughan,Patterson,6,4,3500-5000,2-Storey,43.8797315,-79.467625
1,2019-04-09T00:00:00.000Z,649900.0,2019-05-15T00:00:00.000Z,Sale,N4409516,615000.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '44.31391250000001', 'point': 'PO...","{'area': 'Simcoe', 'zip': 'L0M1B6', 'country':...",634900.0,Sld,U,NaN,NaN,Simcoe,Essa,Essa,Angus,3,3,1500-2000,2-Storey,44.31391250000001,-79.85431349999999
2,2019-05-01T00:00:00.000Z,929900.0,2019-05-15T00:00:00.000Z,Sale,W4434683,885000.0,"{'numKitchens': None, 'numParkingSpaces': '6.0...",ResidentialProperty,"{'latitude': '43.734903', 'point': 'POINT (-79...","{'area': 'Peel', 'zip': 'L6R2H6', 'country': N...",899000.0,Sld,U,NaN,NaN,Peel,Brampton,Brampton,Sandringham-Wellington,4,4,2500-3000,2-Storey,43.734903,-79.762320
3,2019-05-08T00:00:00.000Z,848800.0,2019-05-15T00:00:00.000Z,Sale,W4442890,841000.0,"{'numKitchens': None, 'numParkingSpaces': '6.0...",ResidentialProperty,"{'latitude': '43.6223313', 'point': 'POINT (-7...","{'area': 'Peel', 'zip': 'L4Z3T8', 'country': N...",848800.0,Sld,U,NaN,NaN,Peel,Mississauga,Mississauga,Hurontario,3,3,1500-2000,2-Storey,43.6223313,-79.656504
4,2019-04-24T00:00:00.000Z,692000.0,2019-05-15T00:00:00.000Z,Sale,N4426576,663000.0,"{'numKitchens': None, 'numParkingSpaces': '4.0...",ResidentialProperty,"{'latitude': '43.8893334', 'point': 'POINT (-7...","{'area': 'York', 'zip': 'L4C2X7', 'country': N...",692000.0,Sld,U,NaN,NaN,York,Richmond Hill,Richmond Hill,Crosby,1,3,,Bungalow,43.8893334,-79.43077389999999


Feature Engineering

In [42]:
sale_df = sale_df[sale_df['soldPrice'] != 0]

In [43]:
sale_df.replace({"": np.nan}, inplace = True)
sale_df.replace({float("nan"): np.nan}, inplace = True)

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3173202354.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sale_df.replace({"": np.nan}, inplace = True)
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3173202354.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sale_df.replace({float("nan"): np.nan}, inplace = True)


In [44]:
dfConvertToDatetime(sale_df)

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['listDate'] = df['listDate'].str.slice_replace(start = 10, repl="")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['soldDate'] = df['soldDate'].str.slice_replace(start = 10, repl="")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

In [45]:
createDOM(sale_df)

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DOM'] = dom.astype("string")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(subset=['DOM'], inplace = True)
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\2962372312.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https

In [46]:
sale_df['avg_sqft'] = [(int(value.split("-")[0])+int(value.split("-")[1]))/2 if not isinstance(value, float) and '-' in value else np.nan for value in sale_df['sqft']]

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3629632892.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sale_df['avg_sqft'] = [(int(value.split("-")[0])+int(value.split("-")[1]))/2 if not isinstance(value, float) and '-' in value else np.nan for value in sale_df['sqft']]


In [47]:
sale_df['avg_sqft']

0         4250.0
1         1750.0
2         2750.0
3         1750.0
4            NaN
           ...  
516973       NaN
516974     849.5
516975     649.5
516976     849.5
516977    1099.5
Name: avg_sqft, Length: 516893, dtype: float64

In [48]:
sale_df['ppsqft'] = sale_df['listPrice']/sale_df['avg_sqft']

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3111676264.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sale_df['ppsqft'] = sale_df['listPrice']/sale_df['avg_sqft']


In [49]:
sale_df['ppsqft']

0          352.705882
1          362.800000
2          326.909091
3          485.028571
4                 NaN
             ...     
516973            NaN
516974     806.238964
516975    1307.159353
516976    1059.329017
516977     945.793543
Name: ppsqft, Length: 516893, dtype: float64

In [50]:
sale_df['bathtobed_ratio'] = sale_df['numBathrooms'].astype("Int64")/sale_df['numBedrooms'].astype("Int64")

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\962183113.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sale_df['bathtobed_ratio'] = sale_df['numBathrooms'].astype("Int64")/sale_df['numBedrooms'].astype("Int64")


In [51]:
sale_df['latitude'] = sale_df['latitude'].astype("float")
sale_df['longitude'] = sale_df['longitude'].astype("float")

C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3722179345.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sale_df['latitude'] = sale_df['latitude'].astype("float")
C:\Users\Jackson\AppData\Local\Temp\ipykernel_32192\3722179345.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sale_df['longitude'] = sale_df['longitude'].astype("float")


In [52]:
avg_price_by_area = sale_df.groupby('area').agg('mean')['listPrice']
avg_price_by_city = sale_df.groupby('city').agg('mean')['listPrice']

In [53]:
sale_df_agg = sale_df.join(avg_price_by_area, on = 'area', rsuffix='_by_area')
sale_df_agg = sale_df_agg.join(avg_price_by_city, on = 'city', rsuffix='_by_city')

In [54]:
sale_df_agg.columns

Index(['listDate', 'originalPrice', 'soldDate', 'type', 'mlsNumber',
       'soldPrice', 'details', 'class', 'map', 'address', 'listPrice',
       'lastStatus', 'status', 'raw', 'bathrooms', 'area', 'city', 'district',
       'neighborhood', 'numBathrooms', 'numBedrooms', 'sqft', 'style',
       'latitude', 'longitude', 'DOM', 'avg_sqft', 'ppsqft', 'bathtobed_ratio',
       'listPrice_by_area', 'listPrice_by_city'],
      dtype='object')

In [55]:
columns_to_drop = ['listDate', 'address', 'soldDate', 'type', 'mlsNumber','details', 'map' ,'lastStatus', 'status', 'raw', 'bathrooms']

In [56]:
sale_df_agg = sale_df_agg.drop(columns = columns_to_drop, axis = 1)

In [57]:
#Remove any outliers
sale_df_agg = removeOutliers(sale_df_agg)

In [58]:
sale_df_agg.head()

,originalPrice,soldPrice,class,listPrice,area,city,district,neighborhood,numBathrooms,numBedrooms,sqft,style,latitude,longitude,DOM,avg_sqft,ppsqft,bathtobed_ratio,listPrice_by_area,listPrice_by_city
0,1499000.0,1480000.0,ResidentialProperty,1499000.0,York,Vaughan,Vaughan,Patterson,6,4,3500-5000,2-Storey,43.879731,-79.467625,44,4250.0,352.705882,1.5,1.155431e+06,1.200703e+06
1,649900.0,615000.0,ResidentialProperty,634900.0,Simcoe,Essa,Essa,Angus,3,3,1500-2000,2-Storey,44.313913,-79.854313,36,1750.0,362.800000,1.0,7.612937e+05,7.284088e+05
2,929900.0,885000.0,ResidentialProperty,899000.0,Peel,Brampton,Brampton,Sandringham-Wellington,4,4,2500-3000,2-Storey,43.734903,-79.762320,14,2750.0,326.909091,1.0,9.601132e+05,9.298622e+05
3,848800.0,841000.0,ResidentialProperty,848800.0,Peel,Mississauga,Mississauga,Hurontario,3,3,1500-2000,2-Storey,43.622331,-79.656504,7,1750.0,485.028571,1.0,9.601132e+05,9.390655e+05
4,692000.0,663000.0,ResidentialProperty,692000.0,York,Richmond Hill,Richmond Hill,Crosby,1,3,NaN,Bungalow,43.889333,-79.430774,21,NaN,NaN,0.333333,1.155431e+06,1.224978e+06


Encoding to categorical variables

In [59]:
sale_df_agg_cats = pd.get_dummies(sale_df_agg)

In [60]:
sale_df_agg_cats.shape

(515648, 1865)

In [61]:
sale_df_agg_cats.to_csv(os.getcwd()+"\\data\\sale_data.csv", index = False)